# 02 — Indicadores operacionais e reconciliação com o painel

**Trilha CRISP-DM — modelagem, avaliação e implantação.**

Este notebook reutiliza `analytics/src/indicadores.py` para calcular OTIF, atraso e ocupação, compara com a resposta de `GET /api/v1/admin/kpis` e avalia a sensibilidade da tolerância de OTIF.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "analytics" / "src"))

import indicadores as ind

BANCO = RAIZ / "data" / "svs.db"
conexao = ind.conectar(BANCO)
viagens = ind.carregar_viagens(conexao)
contagens = ind.carregar_contagens(conexao)
conexao.close()
len(viagens), len(contagens)

## 1. Indicadores por data

A tabela abaixo é a mesma saída de `python analytics/src/indicadores.py`.

In [ ]:
por_data = ind.calcular(viagens, contagens)
pd.DataFrame([vars(item) for item in por_data])

In [ ]:
por_veiculo = ind.calcular_por_veiculo(viagens, contagens)
pd.DataFrame([vars(item) for item in por_veiculo])

## 2. Sensibilidade da tolerância de OTIF

OTIF não é um fato objetivo: é uma métrica com parâmetro. Variar a tolerância mostra quanto o indicador depende da política de pontualidade.

In [ ]:
linhas = []
for tolerancia in (0, 5, 10, 15, 30, 60):
    for item in ind.calcular(viagens, contagens, tolerancia):
        linhas.append({"tolerancia_min": tolerancia, "data": item.data, "otif": item.otif_percentual})

sensibilidade = pd.DataFrame(linhas)
sensibilidade.pivot(index="data", columns="tolerancia_min", values="otif")

## 3. Reconciliação com o painel

Com a API em execução (`make run`), compare o dia mais recente com `GET /api/v1/admin/kpis`. Pequenas diferenças são esperadas apenas se a data virar entre as execuções ou se houver telemetria nova.

In [ ]:
import urllib.request

CHAVE = "dev-admin-key"  # apenas ambiente local de demonstração
URL = "http://localhost:8000/api/v1/admin/kpis"

try:
    requisicao = urllib.request.Request(URL, headers={"X-Admin-Key": CHAVE})
    with urllib.request.urlopen(requisicao, timeout=5) as resposta:
        kpis = json.load(resposta)
    exibicao = {k: kpis[k] for k in ("data", "viagens_total", "viagens_em_andamento", "otif_percentual", "atraso_medio_min", "passageiros_embarcados")}
    print(json.dumps(exibicao, ensure_ascii=False, indent=2))
except Exception as exc:
    print("API indisponível; execute `make run` para reconciliar.", exc)

In [ ]:
if por_data:
    ultimo = por_data[-1]
    print(ind.resumo_txt(ultimo))
else:
    print("Sem dados.")

## 4. Implantação

O produto desta análise é operacional:

1. Exportar o CSV com `indicadores.py`.
2. Comparar com o painel no mesmo dia.
3. Registrar a leitura no relatório, explicitando tolerância e denominadores.
4. Nunca versione o CSV com dados operacionais; trate-o como artefato local.

Para produção, substitua a conexão SQLite por uma réplica de leitura PostgreSQL e agende a execução. A fórmula deve permanecer única: se mudar, atualize painel, script, README e testes.